<a href="https://colab.research.google.com/github/Nox612/Project_CNN/blob/main/cnn_cats_dogs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Contents**
1. Setup
2. Dataset exploration
3. Build the CNN
4. Train (learning curves, overfitting/underfitting diagnosis)
5. Evaluate (confusion matrix, precision/recall/F1)
6. Improve the model (augmentation; **+ augment-validation experiment**; dropout+BN; combined)
7. Hyper-parameter experiments (filters, learning rate, optimizer, image size, head, dense)
8. Bonus: transfer learning with MobileNetV2 (We did not do this)


## 1. Setup — configuration, seeds, determinism and GPU check

We make `src/` importable, load configuration from `.env`, seed everything and
**enable TensorFlow op determinism** (inside `config.set_seeds`) so results are
reproducible. We also start a fresh `results/experiments.csv` so the file is
written purely by this notebook run.

In [ ]:
# Make the reusable src/ modules importable and load configuration from .env.
import sys, os, json
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # quiet TF C++ logs (set before importing TF)
PROJECT_DIR = r"/content/drive/MyDrive/cnn_cats_dogs"
sys.path.append(os.path.join(PROJECT_DIR, "src"))

import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import Image, display

import config                 # typed config values + seeds/determinism/GPU check
import data                   # dataset extraction, integrity, tf.data pipeline
import models as M            # model factories (baseline CNN + MobileNetV2)
import training as T          # fit wrapper, learning-curve plots, CSV logger
import evaluation as ev       # predictions, metrics, confusion matrix, errors
import experiment as X        # orchestration: run one experiment, sweep, figures

config.set_seeds()            # seeds Python/NumPy/TF AND enables op determinism
config.check_gpu()            # warns if there is no GPU (switch Colab to T4 GPU)

# Start a clean experiment log so this run is the single source of truth.
if config.EXPERIMENTS_CSV.exists():
    config.EXPERIMENTS_CSV.unlink()

# Collect the test-set metrics of every model we evaluate, for the final table.
test_metrics = []

print("TensorFlow:", tf.__version__)
print("Config:", config.config_summary())

ModuleNotFoundError: No module named 'config'

In [ ]:
# Small helper: display a figure that a src helper saved to figures/. We show the
# saved PNG (rather than re-plotting) so the image in the notebook is byte-for-byte
# the same file embedded in the report.
def show_fig(*names):
    for n in names:
        p = config.FIGURES_DIR / n
        if p.exists():
            display(Image(filename=str(p)))

## 2. Dataset exploration

In [ ]:
# Ensure the data is on local disk (re-extracts from the course zip if missing).
dataset_root = data.ensure_dataset()
print("Dataset root:", dataset_root)

# Count images per split and class.
counts = {}
for split, directory in [("train", data.TRAIN_DIR), ("validation", data.VALIDATION_DIR)]:
    for class_name in config.CLASS_NAMES:
        counts[(split, class_name)] = len(list((directory / class_name).glob("*.jpg")))

n_train = counts[("train", "cats")] + counts[("train", "dogs")]
n_val = counts[("validation", "cats")] + counts[("validation", "dogs")]
print("Per-class counts:", counts)
print(f"train/ total = {n_train}   validation/ total = {n_val}")

**Integrity check.** Before training we confirm every file is a decodable image —
a corrupt or non-image file would crash the `tf.data` pipeline mid-training.

In [ ]:
# Decode-check every file in both folders and report any corrupt / non-image files.
for split, directory in [("train", data.TRAIN_DIR), ("validation", data.VALIDATION_DIR)]:
    report = data.check_integrity(directory)
    print(f"[{split}] files={report['n_files']}  "
          f"corrupt={report['n_corrupt']}  non_image={report['n_non_image']}")

**Class distribution and input shapes**

In [ ]:
train_paths, train_labels, test_paths, test_labels = data.stratified_train_test_split()
val_paths, val_labels = data.list_files_labels(data.VALIDATION_DIR)
splits = {"train": (train_paths, train_labels),
          "val": (val_paths, val_labels),
          "test": (test_paths, test_labels)}

size_stats = X.make_exploration_figures(splits)
print("Raw image size stats (pixels):", size_stats)
show_fig("class_distribution.png", "size_distribution.png")

In [ ]:
# A grid of random labelled training images gives a feel for the task difficulty.
show_fig("sample_grid.png")

In [ ]:

ds = data.build_datasets(img_size=config.IMG_SIZE, batch_size=config.BATCH_SIZE)
train_ds, val_ds, test_ds = ds["train_ds"], ds["val_ds"], ds["test_ds"]
test_paths = ds["splits"]["test"][0]   # file paths, for showing misclassified images

print(f"train = {len(ds['splits']['train'][0])} images "
      f"(cats/dogs = {np.bincount(ds['splits']['train'][1])})")
print(f"test  = {len(ds['splits']['test'][0])} images "
      f"(cats/dogs = {np.bincount(ds['splits']['test'][1])})")
print(f"val   = {len(ds['splits']['val'][0])} images "
      f"(cats/dogs = {np.bincount(ds['splits']['val'][1])})")

## 3. Build the CNN

**Architecture (baseline).** Four convolutional blocks with a growing number of
3×3 filters, each followed by 2×2 max pooling, then a dense head. With `valid`
padding each 3×3 conv shrinks the map by 2 px before the 2× pool, so the spatial
size goes 150 → 148 → 74 → 72 → 36 → 34 → 17 → 15 → 7.

| Block | Layer | Output shape |
|------|-------|--------------|
| input | — | 150×150×3 |
| 1 | Conv2D(32, 3×3, ReLU) → MaxPool2D | 74×74×32 |
| 2 | Conv2D(64, 3×3, ReLU) → MaxPool2D | 36×36×64 |
| 3 | Conv2D(128, 3×3, ReLU) → MaxPool2D | 17×17×128 |
| 4 | Conv2D(128, 3×3, ReLU) → MaxPool2D | 7×7×128 |
| head | Flatten → Dense(512, ReLU) → Dense(1, sigmoid) | 1 |

**Compile choices** This is a two-class problem, so the output is a
**single sigmoid** unit trained with **binary cross-entropy**; the natural
pairing for binary classification (a probability that reads directly
as `P(dog)`). The alternative, a 2-way **softmax + categorical cross-entropy**, is
mathematically equivalent for two classes but uses an extra output unit and
one-hot labels; sigmoid/BCE is leaner, so we use it.

 Optimizer: **Adam**
(`lr=1e-3`); a robust adaptive default that converges fast with little tuning
(Step 7 sweeps SGD/RMSprop and other learning rates)



In [ ]:
from tensorflow.keras import layers, models as kmodels, optimizers

def build_baseline_cnn(img_size=config.IMG_SIZE):
    """The reference 4-conv-block CNN"""
    inputs = layers.Input(shape=(img_size, img_size, 3), name="input")
    x = layers.Rescaling(1.0 / 255, name="rescale")(inputs)

    x = layers.Conv2D(32,  (3, 3), activation="relu", name="conv1")(x)
    x = layers.MaxPooling2D((2, 2), name="pool1")(x)
    x = layers.Conv2D(64,  (3, 3), activation="relu", name="conv2")(x)
    x = layers.MaxPooling2D((2, 2), name="pool2")(x)
    x = layers.Conv2D(128, (3, 3), activation="relu", name="conv3")(x)
    x = layers.MaxPooling2D((2, 2), name="pool3")(x)
    x = layers.Conv2D(128, (3, 3), activation="relu", name="conv4")(x)
    x = layers.MaxPooling2D((2, 2), name="pool4")(x)

    x = layers.Flatten(name="flatten")(x)
    x = layers.Dense(512, activation="relu", name="fc")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="output")(x)

    model = kmodels.Model(inputs, outputs, name="baseline")
    model.compile(optimizer=optimizers.Adam(learning_rate=config.LEARNING_RATE),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model

baseline = build_baseline_cnn()
baseline.summary()

### Parameter count - by hand


| Layer | Formula | Params |
|------|---------|-------:|
| conv1 | (3·3·3 + 1)·32 | 896 |
| conv2 | (3·3·32 + 1)·64 | 18,496 |
| conv3 | (3·3·64 + 1)·128 | 73,856 |
| conv4 | (3·3·128 + 1)·128 | 147,584 |
| pooling ×4 | no parameters | 0 |
| flatten | 7·7·128 = 6272 units | 0 |
| fc | (6272 + 1)·512 | 3,211,776 |
| output | (512 + 1)·1 | 513 |
| **Total** | | **3,453,121** |


In [ ]:
# Reproduce the by-hand calculation in code and assert it matches Keras exactly.
conv1 = (3 * 3 * 3 + 1) * 32
conv2 = (3 * 3 * 32 + 1) * 64
conv3 = (3 * 3 * 64 + 1) * 128
conv4 = (3 * 3 * 128 + 1) * 128
fc = (7 * 7 * 128 + 1) * 512
output = (512 + 1) * 1
hand_total = conv1 + conv2 + conv3 + conv4 + fc + output

print("conv1..conv4:", conv1, conv2, conv3, conv4)
print("fc, output  :", fc, output)
print("hand total  :", hand_total)
print("Keras total :", baseline.count_params())
assert hand_total == baseline.count_params(), "parameter mismatch!"
print("match ✓")

## 4. Train the baseline

We train for **15 epochs** with **batch size 100** (both from `.env`), tracking
train/validation accuracy and loss.

In [ ]:
config.set_seeds()
baseline_run = X.run_experiment(
    run_id="baseline",
    what_changed="4-conv CNN, no regularisation (reference)",
    model=baseline, train_ds=train_ds, val_ds=val_ds, epochs=config.EPOCHS,
    meta={}, test_ds=test_ds, test_paths=test_paths, save_model=True, verbose=2)

show_fig("baseline_curves.png")

**Diagnosis - overfitting.** The training accuracy keeps climbing into the low
0.90s while the validation accuracy plateaus in the mid-0.70s, and the validation
With only 1600 training images and
~3.45M parameters, the model has more than enough
capacity to memorise training-specific detail that does not generalise.

## 5. Evaluate on the held-out test set
The predictions were
computed inside `run_experiment` above.

In [ ]:
print("Baseline test metrics:",
      {k: round(v, 3) for k, v in baseline_run["test"].items() if k != "run_id"})
test_metrics.append(baseline_run["test"])
show_fig("baseline_confusion.png")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
y_true, y_pred, y_prob = baseline_run["predictions"]
print(classification_report(y_true, y_pred, target_names=config.CLASS_NAMES, digits=3))

cm = confusion_matrix(y_true, y_pred)
cats_as_dog, dogs_as_cat = int(cm[0, 1]), int(cm[1, 0])
print(f"cats misclassified as dogs: {cats_as_dog}")
print(f"dogs misclassified as cats: {dogs_as_cat}")
print("-> the baseline confuses", "dogs" if dogs_as_cat > cats_as_dog else "cats", "more often")

In [ ]:
show_fig("baseline_misclassified.png")

**Interpretation:** Overall accuracy sits in the mid-0.7s

The baseline has **high precision but lower recall on the
"dog" class**, it is *conservative about predicting "dog"*. As the printed
counts and the confusion matrix show, most of its errors are therefore **dogs
mislabelled as cats**; dogs are the more-confused class here. A plausible reason:
"dog" is visually far more diverse than "cat" (large variation in breed, size,
snout and coat), so from only 1600 images the model's "dog" concept is fuzzier and
it defaults to the safer "cat" prediction on ambiguous images. Looking at the
misclassified grid, those hard cases are typically low-contrast animals, unusual
poses, tight close-ups where the background dominates, or cluttered scenes.

## 6. Improve the model

Each technique is tested on its own and compared to the baseline. To keep it fair, every model trains for the same 15 epochs as the baseline
### Should the validation set be augmented?


In [ ]:
# Visualise the augmentation transforms we apply (random flip/rotation/zoom/shift/contrast).
show_fig("augmentation_examples.png")

In [ ]:

config.set_seeds()
aug_model = M.build_cnn(augment=True, name="data_augmentation")
aug_run = X.run_experiment(
    run_id="data_augmentation",
    what_changed="train-only augmentation (flip/rotation/zoom/shift/contrast)",
    model=aug_model, train_ds=train_ds, val_ds=val_ds, epochs=config.IMPROVE_EPOCHS,
    meta={"augment": True}, test_ds=test_ds, test_paths=test_paths, save_model=True,
    verbose=2)
test_metrics.append(aug_run["test"])
show_fig("data_augmentation_curves.png")

In [ ]:

config.set_seeds()
val_ds_aug = data.augment_dataset(val_ds, M.augmentation_block(config.SEED))
augval_model = M.build_cnn(augment=True, name="aug_trainval")
augval_run = X.run_experiment(
    run_id="aug_trainval",
    what_changed="augment BOTH train and validation (brief's literal wording)",
    model=augval_model, train_ds=train_ds, val_ds=val_ds_aug, epochs=config.IMPROVE_EPOCHS,
    meta={"augment": True}, test_ds=test_ds, test_paths=test_paths, save_model=False,
    verbose=2)
test_metrics.append(augval_run["test"])
show_fig("aug_trainval_curves.png")

In [ ]:
# Direct comparison: train-only vs train+validation augmentation.
augmentation_compare = pd.DataFrame([
    {"model": r["row"]["run_id"],
     "test_acc": round(r["test"]["accuracy"], 3),
     "best_val_acc": round(r["row"]["best_val_acc"], 3),
     "val_loss": round(r["row"]["val_loss"], 3),
     "train_time_s": r["row"]["train_time_s"]}
    for r in [aug_run, augval_run]])
augmentation_compare

**Train-only vs train+validation augmentation.** Both models augment the training
set, and the **test set is clean in both**, so their **test accuracy is
comparable**


In [ ]:
# Dropout(0.5) + BatchNorm only (no augmentation)
config.set_seeds()
dbn_model = M.build_cnn(dropout=0.5, batch_norm=True, name="dropout_bn")
dbn_run = X.run_experiment(
    run_id="dropout_bn",
    what_changed="dropout(0.5) + batch-norm",
    model=dbn_model, train_ds=train_ds, val_ds=val_ds, epochs=config.IMPROVE_EPOCHS,
    meta={"dropout": 0.5, "batch_norm": True}, test_ds=test_ds, test_paths=test_paths,
    save_model=True, verbose=2)
test_metrics.append(dbn_run["test"])
show_fig("dropout_bn_curves.png")

In [ ]:
# Combined: augmentation + dropout + batch-norm (stack both regularisers).
config.set_seeds()
combined_model = M.build_cnn(augment=True, dropout=0.5, batch_norm=True, name="combined")
combined_run = X.run_experiment(
    run_id="combined",
    what_changed="augmentation + dropout(0.5) + batch-norm",
    model=combined_model, train_ds=train_ds, val_ds=val_ds, epochs=config.IMPROVE_EPOCHS,
    meta={"augment": True, "dropout": 0.5, "batch_norm": True},
    test_ds=test_ds, test_paths=test_paths, save_model=True, verbose=2)
test_metrics.append(combined_run["test"])
show_fig("combined_curves.png")

In [ ]:
# Compare the from-scratch models on the held-out test set.
improve_table = pd.DataFrame([
    {"model": r["test"]["run_id"], **{k: round(v, 3) for k, v in r["test"].items() if k != "run_id"},
     "train_acc": round(r["row"]["train_acc"], 3), "best_val_acc": round(r["row"]["best_val_acc"], 3),
     "gap(train-val)": round(r["row"]["train_acc"] - r["row"]["best_val_acc"], 3)}
    for r in [baseline_run, aug_run, augval_run, dbn_run, combined_run]])
improve_table

## 7. Hyper-parameter experiments

changing **one thing at a
time** versus the baseline: filter counts, learning rate, optimizer, image size,
Flatten vs GlobalAveragePooling, and dense width. To respect limited GPU time
each run is short (**12 epochs, early stopping, no augmentation**), so the
absolute numbers sit a little below a fully-trained model

In [ ]:

X.run_sweep(ds, max_epochs=12, patience=4)
print("sweep complete")

In [ ]:
exp = pd.read_csv(config.EXPERIMENTS_CSV)
display(exp[["run_id", "what_changed", "val_acc", "best_val_acc", "val_loss",
             "train_acc", "train_time_s", "param_count"]].round(3))

The learning rate had by far the biggest effect. At 0.01 training diverged and the model stayed at chance level (50%). At 0.0001 it was stable but learned too slowly to reach a good score in 12 epochs (70.2%). That confirms 0.001 as the right value. The optimizer mattered for the same reason: at this learning rate, SGD with momentum barely learned (51.1%), while RMSprop (72.1%) worked but stayed slightly behind Adam, whose step size adapts for each weight. Adding capacity did not help. Doubling the filters to 32-64-128-256 doubled the parameter count (6.8M) without improving accuracy (74.3%), and halving them cost only a few points (70.6%). With only 1,600 training images, the amount of data, not the size of the model, limits performance. Shrinking the classifier part of the network confirmed this. A Dense layer of 256 units instead of 512 gave the same accuracy (74.0%) with half the parameters, and replacing Flatten with GlobalAveragePooling reduced the model from 3.45M to 0.31M parameters, eleven times fewer, for a loss of only a few points (71.5%). Finally, an image size of 128 pixels lost useful detail (72.1%), and 224 pixels tripled the parameter count and gave a worse result (68.9%), which confirms that 150 pixels is the right compromise.

## Overall comparison and conclusions

In [ ]:
# Headline comparison across every model we evaluated on the test set (+ transfer head and transfer finetune)
X.make_comparison_figure(test_metrics)
show_fig("model_comparison.png")

final_table = pd.DataFrame(
    [{"model": t["run_id"], **{k: round(v, 3) for k, v in t.items() if k != "run_id"}}
     for t in test_metrics]).sort_values("accuracy", ascending=False)
final_table